# పాఠం 05 - ఏజెంటిక్ RAG


## సెటప్

ఈ నోట్బుక్ మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్‌ను ఉపయోగించి ఏజెంటిక్ RAG (రిట్రీవల్-ఆగ్మెంటెడ్ జనరేషన్) ప్యాటర్న్‌ను ప్రదర్శిస్తుంది.

**ముందస్తు అవసరాలు:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — మీ Azure AI సర్చ్ సర్వీస్ ఎండ్‌పాయింట్
- `AZURE_SEARCH_API_KEY` — మీ Azure AI సర్చ్ API కీ
- ఎన్విరాన్‌మెంట్ వేరియబుల్స్ ద్వారా కాన్ఫిగర్ చేసిన Azure OpenAI డిప్లాయ్‌మెంట్
- Azure CLI ధృవీకరించబడింది (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ఏజెన్సిక్ RAG అంటే 무엇인가?

సాంప్రదాయ RAG ఒక నిర్ధారిత పైప్‌లైన్‌ను అనుసరిస్తుంది: డాక్యుమెంట్లు రాబట్టి, ఆపై సమాధానం తయారు చేయడం. **ఏజెన్సిక్ RAG** ఆజెంట్‌కు స్వతంత్రతను ఇస్తుంది, అంటే **ఎప్పుడు** మరియు **ఎలా** సమాచారం రాబట్టాలో నిర్ణయించుకోవచ్చు.

ఏజెన్సిక్ RAG ద్వారా, ఆజెంట్ చేయగలదు:
- ప్రశ్నకు సమాధానం ఇచ్చే ముందు రాబట్టడం అవసరమా లేదో **నిర్ణయించు**
- ఏ డేటా మూలం లేదా పరికరాన్ని విచారించాలో **ఎంచుకోండి**
- రాబట్టిన ఫలితాలను **మూల్యాంకనం చేయండి** మరియు మొదటి ప్రయత్నం తక్కువైతే మరల రాబట్టడాన్ని చేయండి
- అనేక రాబట్టే దశల నుంచి సమాచారాన్ని కలిపి సమగ్ర సమాధానంగా **కలిపి ఇవ్వండి**

ఇది ఆజెంట్‌ను ఒక స్థిరమైన రాబట్టి-తర్వాత-తయారు చేసే పైప్‌లైన్‌ కంటే ఎక్కువ సునిశితంగా మరియు సరైనదిగా చేస్తుంది.


## శోధన సాధనం సృష్టించడం

Agentic RAGలో, బయటి డేటా మూలాలు ఏజెంట్ డిమాండ్ పై పిలవగల **సాధనాలు** గా మూసివేయబడ్డాయి. దీనివల్ల ఏజెంట్ రిట్రీవల్‌ను ఒక్క చర్యగా ఉన్నదిగా పరిగణించవచ్చు, తప్పనిసరి దశగా కాదు.

క్రింద మేము ఒక ప్రయాణ జ్ఞానాధారాన్ని నిర్వచించి, ఏజెంట్ పిలవగల సాధనంగా అందిస్తున్నాము గమ్యస్థాన సమాచారాన్ని చూడడానికి.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ఏజెంట్ను నిర్మించడం

ఇప్పుడ 우리는 항상 답변하기 전에 정보를 검색하도록 지시된 에이전트를 생성합니다. 에이전트는 자신의 훈련 데이터에 의존하는 대신 지식 기반에 응답을 고정하기 위해 `search_travel_knowledge` 도구를 사용합니다.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## పునరావృత రిట్రీవల్ — మేకర్-చెక్కర్ నమూనా

Agentic RAG యొక్క ప్రధాన ప్రయోజనం **పునరావృత రిట్రీవల్**. ఏజెంట్ ஆரம்பిక కనుగొనింపులపై ధ్రువీకరించడం, మెరుగుపరచడం లేదా విస్తరించడానికి ఒకাধিক సార్లు శోధన చేయొచ్చు — "మేకర్-చెక్కర్" వర్క్‌ఫ్లోకు సారూప్యంగా:

1. **మేకర్ దశ**: ఏజెంట్ ప్రారంభ సమాచారం రిట్రీవ్ చేసి ప్రతిస్పందనను డ్రాఫ్ట్ చేస్తుంది.
2. **చెక్కర్ దశ**: ఏజెంట్ వివరాలను ధ్రువీకరించడానికి లేదా గ్యాప్స్ నింపేందుకు అదనపు రిట్రీవల్స్ చేస్తుంది.

క్రింద, ఏజెంట్ అనేక ప్రాంతాలను ਤੁలించాల్సిన ప్రశ్న అడిగింది, ఇది దానికి అనేక పెళ్ళలు శోధించమని ప్రేరేపిస్తుంది.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి **Agentic RAG** వ్యవస్థను ఎలా నిర్మించాలో నేర్చుకున్నారు:

- **Agentic RAG** ఏజెంట్లు సమాచారం ఎప్పుడు తీసుకోవాలో స్వతంత్రంగా నిర్ణయించేందుకు అనుమతిస్తుంది, అందువలన తీసుకోవడం స్థిరంగా కాకుండా డైనమిక్ అవుతుంది.
- **పరికరాలను డేటా మూలాలుగా ఉపయోగించడం**: బాహ్య జ్ఞాన ఆధారాలు (Azure AI Search వంటి) ఏజెంట్ వాడుకోవచ్చిన పరికరాలుగా ర్యాప్ చేయబడతాయి.
- **పునఃపరిశీలన తీసుకోను**: మేకర్-చెకర్ నమూనా ఏజెంట్‌కు అనేక రౌండ్లు అందుబాటులో ఉండేందుకు, అన్వేషణ, ధృవీకరణ, మరియు పర్యవేక్షణ చేయడానికి వీలు కల్పిస్తుంది, తద్వారా తుది సమాధానం వస్తుంది.

ఉత్పత్తిలో, మీరు లో-మెమోరి `TRAVEL_KNOWLEDGE_BASE` ను పెద్ద పరిమాణం ప్రయాణ పత్రాల తీసుకునే పని కోసం నిజమైన Azure AI Search సూచికతో మార్చాలి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**దృష్టాంతం**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించారు. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, స్వయంచాలక అనువాదాల్లో పొరపాట్లు లేదా తప్పిదాలు ఉండవచ్చు. అసలు పత్రం స్థానిక భాషలో ఉన్నది అధికారిక మూలం అని పరిగణించాలి. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం తీసుకోవడం మంచిది. ఈ అనువాదాన్ని ఉపయోగించడం వల్ల ఏర్పడే ఏవైనా తప్పిప్రతిపాదనలు లేదా అపార్థాలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
